# Coffee17 Runtime Head-to-Head V1
Architecture-only CUDA benchmark: B0 / W3 / C2 / B1 / W4. No training, no dataset, no checkpoints, no outer test. Enable one Kaggle GPU and Internet.

In [ ]:
import os, sys, subprocess, tempfile, importlib.util
from pathlib import Path

PIN = "b659a22a72c97bfe75576424a38cbb10076ab0be"
WORK = Path("/kaggle/working")
assert WORK.is_dir(), "Run in Kaggle."
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan GPU Kaggle. Tidak ada CPU fallback.")
print("TRAINING: DISABLED")
print("DATASET: NOT REQUIRED")
print("OUTER TEST: LOCKED")
print("SCIENTIFIC PIN:", PIN)
print("GPU:", torch.cuda.get_device_name(0))

# Isolated disposable source checkout, not the training/output directory.
REPO = Path(tempfile.mkdtemp(prefix="coffee17-runtime-v1-")) / "repo"
subprocess.run(["git", "clone", "--quiet", "--no-checkout",
                "https://github.com/ediprin/coffee-bean-classification.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", "--detach", PIN], check=True)
# Preserve the existing CUDA/PyTorch stack; install missing timm only if necessary.
if importlib.util.find_spec("timm") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "timm==1.0.20"], check=True)
ENV = dict(os.environ)
ENV["PYTHONPATH"] = str(REPO / "src") + os.pathsep + ENV.get("PYTHONPATH", "")
CONFIG = REPO / "configs/frequency_screening/FOCUSED_EFFICIENCY_V1.yaml"
RUNNER = REPO / "src/bilinear_lmmd/experiments/profile_runtime_head_to_head_v1.py"
compile(RUNNER.read_text(), str(RUNNER), "exec")
# Lightweight helper assertions, no model training and no data loader.
CHECK = """
from bilinear_lmmd.experiments.profile_runtime_head_to_head_v1 import ARMS, balanced_orders, stats
orders = balanced_orders()
assert len(orders) == 10
assert all(sorted(order) == sorted(ARMS) for order in orders)
assert all(sum(order[i] == a for order in orders) == 2 for a in ARMS for i in range(len(ARMS)))
assert stats([1,2,3,4,5])['median'] == 3
assert stats(list(range(1,101)))['p95'] == 95
print('HELPER CHECKS: PASS')
"""
subprocess.run([sys.executable, "-c", CHECK], cwd=REPO, env=ENV, check=True)


In [ ]:
import json
OUT = WORK / "coffee17-runtime-head-to-head-v1.json"
# Do not overwrite a prior successful benchmark; use a new output name if needed.
if OUT.exists():
    raise FileExistsError(f"Existing result preserved: {OUT}. Download it or rename it before repeating.")
subprocess.run([
    sys.executable, "-u", "-m", "bilinear_lmmd.experiments.profile_runtime_head_to_head_v1",
    "--config", str(CONFIG),
    "--required-commit", PIN,
    "--device", "cuda:0",
    "--warmup", "30", "--block-warmup", "10", "--samples", "50",
    "--output", str(OUT),
], cwd=REPO, env=ENV, check=True)
result = json.loads(OUT.read_text())
assert result["training_performed"] is False
assert result["dataset_accessed"] is False
assert result["outer_test_accessed"] is False
print("\nARM       PARAMS      MEDIAN(ms)    P95(ms)    IMG/S     PEAK(MiB)")
for r in result["results"]:
    print(f"{r['candidate']:4} {r['parameter_count']:12,d} "
          f"{r['latency_ms']['median']:12.3f} {r['latency_ms']['p95']:10.3f} "
          f"{r['sustained_batch1_img_s']:9.2f} "
          f"{r['isolated_memory']['model_input_and_forward_peak_bytes']/2**20:10.2f}")
print("\nUPLOAD:", OUT)
print("NO TRAINING. NO CHECKPOINT DELETION.")
